In [ ]:
# nanoGPT from scratch on Kaggle
# 第 1 个 cell：环境检查、目标说明、导入依赖、固定随机种子
#
# 这份 notebook 的目标不是直接调用官方 nanoGPT 的 train.py，
# 而是在 Kaggle 里一步一步“自己写出一个 nanoGPT”。
# 官方源码已经放在仓库的 nanoGPT-reference/ 目录中，后续每一步都会对照它来实现。
#
# 这份 notebook 的整体路线：
# Cell 2：按官方 GPT-2 复现路线准备 OpenWebText，并用 GPT-2 BPE 变成 token
# Cell 3：写 get_batch，构造 x/y 训练样本
# Cell 4：实现单头 causal self-attention
# Cell 5：实现多头注意力
# Cell 6：实现 MLP 和 Block
# Cell 7：组装 GPTConfig 和 GPT
# Cell 8：写训练循环，跑多轮训练，并在 notebook 里启用可用的多卡加速
# Cell 9：写 generate，输出验证
# 当前第 1 格只做准备工作：
# 1. 确认 PyTorch 和 GPU 状态
# 2. 处理 Kaggle 上 P100 + 新版 PyTorch 不兼容的问题
# 3. 导入后续会用到的基础库
# 4. 固定随机种子，方便复现实验
# 5. 设置一个 device 变量，后续张量和模型都放到这个设备上

import math
import os
import random
from dataclasses import dataclass

# 这个环境变量用于缓解 PyTorch CUDA caching allocator 的显存碎片问题。
# 必须在 import torch 之前设置才最有效；如果你已经运行过 torch，再改它通常不会完全生效。
# Kaggle 上如果频繁中断/重跑训练 cell，建议重启 kernel 后从 Cell 1 开始运行。
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F


print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


def choose_device():
    # 选择当前 Kaggle 环境里真正能安全使用的设备。
    # 为什么不只看 torch.cuda.is_available()？
    # 因为某些 Kaggle 运行时会出现“能看到 GPU，但当前 PyTorch wheel 不支持这张卡架构”的情况。
    # 例如 P100 是 sm_60，而较新的 CUDA wheel 可能只编译了 sm_70 以上 kernel。
    # 这时如果盲目返回 cuda，前几行能跑，真正做矩阵乘法时才会崩。
    if not torch.cuda.is_available():
        print('GPU not found, using CPU. 也能跑小实验，但会慢一些。')
        return 'cpu'

    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print('GPU:', name)
    print('CUDA capability:', f'sm_{props.major}{props.minor}')

    # Kaggle 有时会分配 Tesla P100。P100 是 sm_60。
    # 如果当前 PyTorch 构建不支持 sm_60，torch.cuda.is_available() 仍可能是 True，
    # 但真正执行 CUDA kernel 时会失败，所以这里主动退回 CPU。
    if props.major < 7:
        print()
        print('注意：当前 GPU 是 P100/sm_60，但这个 PyTorch 构建可能不支持 sm_60。')
        print('本 notebook 将临时使用 CPU。想用 GPU，请在 Kaggle Accelerator 里优先选择 T4。')
        print('如果 Kaggle 只能给 P100，也可以改装兼容旧架构的 PyTorch，但初学阶段先不折腾环境。')
        print()
        return 'cpu'

    return 'cuda'


device = choose_device()


# 固定随机种子。
# 初始化权重、打乱数据、采样 token 都会用到随机数。
# 固定 seed 后，每次从头运行 notebook，结果更容易复现和对比。
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device == 'cuda':
    torch.cuda.manual_seed(seed)


# 允许 TensorFloat-32。它是 NVIDIA GPU 上的一种加速格式。
# 对学习实验来说，可以让矩阵乘法更快。
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


print('Using device:', device)
print('Seed:', seed)




In [ ]:
# 第 2 个 cell：官方 OpenWebText + GPT-2 BPE 数据准备
#
# 这一格对应 nanoGPT 官方源码：
#   nanoGPT-reference/data/openwebtext/prepare.py
#
# 你提到的“基于 openwebtext dataset 复现 GPT-2”，对应的就是这条路线：
#
#   OpenWebText -> GPT-2 BPE tokenizer -> train.bin / val.bin
#
# 注意：完整 OpenWebText 很大。nanoGPT 官方注释里写到：
#   Hugging Face cache 约 54GB
#   train.bin 约 17GB
#   val.bin 约 8.5MB
#   train 约 9B tokens
#
# 所以这一格优先使用 Kaggle Input 里已经准备好的 train.bin / val.bin。
# 如果你确认 Kaggle 磁盘和时间足够，再把 PREPARE_OPENWEBTEXT_FROM_HF 改成 True，
# 它会按官方 prepare.py 的方式从 Hugging Face 下载并预处理完整 OpenWebText。

import os
import subprocess
import sys
from pathlib import Path

import numpy as np


PREPARE_OPENWEBTEXT_FROM_HF = False


def ensure_package(import_name, pip_name=None):
    # 缺少依赖时自动安装，方便在 Kaggle 新环境中直接运行。
    # import_name 是 Python 里 import 用的名字，pip_name 是 pip install 用的包名。
    # 多数时候两者一样，例如 import tiktoken / pip install tiktoken。
    # 有些库两者不同，就可以单独传 pip_name。
    try:
        return __import__(import_name)
    except ModuleNotFoundError:
        package_name = pip_name or import_name
        print(f'当前环境没有 {package_name}，正在安装。')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        return __import__(import_name)


tiktoken = ensure_package('tiktoken')
ensure_package('tqdm')
from tqdm import tqdm


# GPT-2 BPE tokenizer。
# BPE 可以理解成“常见文本片段优先合并”的分词方式。
# GPT-2 的普通词表大小是 50257，其中结束符通常是 50256。
# nanoGPT 从零训练 GPT-2 规模模型时，默认把 vocab_size padding 到 50304，
# 这样矩阵维度更适合 GPU 计算。
enc = tiktoken.get_encoding('gpt2')
gpt2_vocab_size = enc.n_vocab
vocab_size = 50304

print('Tokenizer:', enc.name)
print('GPT-2 原始词表大小:', gpt2_vocab_size)
print('GPT-2 eot_token:', enc.eot_token)
print('nanoGPT 从零训练默认 vocab_size:', vocab_size)


example_text = 'The quick brown fox jumps over the lazy dog.'
example_ids = enc.encode_ordinary(example_text)

print()
print('BPE 编码/解码小例子：')
print('原始文本:', example_text)
print('编码后 token id:', example_ids)
print('解码回去:', enc.decode(example_ids))
print('每个 token id 对应的文本片段：')
for token_id in example_ids:
    print(f'{token_id:>6} -> {enc.decode([token_id])!r}')


def find_prepared_openwebtext_bins():
    # 优先在 Kaggle Input 里找已经准备好的 OpenWebText 二进制 token 文件。
    # 这样不占 /kaggle/working 的 Output 空间。
    # Kaggle Input 是只读挂载，适合放大数据；Output 才会计入你本次 notebook 的输出空间。
    # 所以这里先找现成 train.bin / val.bin，而不是一上来就下载几十 GB 原始数据。
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.exists():
        return None

    train_candidates = sorted(kaggle_input.rglob('train.bin'))
    val_candidates = sorted(kaggle_input.rglob('val.bin'))

    for train_path in train_candidates:
        same_dir_val = train_path.with_name('val.bin')
        if same_dir_val.exists():
            return train_path, same_dir_val

    if train_candidates and val_candidates:
        return train_candidates[0], val_candidates[0]

    return None


def prepare_openwebtext_from_hf():
    # 这部分严格对应 nanoGPT 官方 data/openwebtext/prepare.py。
    # 完整运行会下载和处理大量数据，Kaggle 免费环境可能因为空间或时间失败。
    # 只有当你明确把 PREPARE_OPENWEBTEXT_FROM_HF 改成 True 时才会走这里。
    # 这相当于从“直接用别人准备好的食材”切换到“自己买菜、洗菜、切菜”。
    # 结果格式一样，都是 GPT-2 BPE token 的 train.bin / val.bin，但耗时和磁盘占用完全不同。
    ensure_package('datasets')
    from datasets import load_dataset

    if Path('/kaggle/working').exists():
        data_dir = Path('/kaggle/working/data/openwebtext')
    else:
        data_dir = Path('data/openwebtext')
    data_dir.mkdir(parents=True, exist_ok=True)

    num_proc = min(8, max(1, (os.cpu_count() or 2) // 2))
    num_proc_load_dataset = num_proc

    print('开始加载 OpenWebText。第一次运行会下载大量数据。')
    dataset = load_dataset('openwebtext', num_proc=num_proc_load_dataset)

    split_dataset = dataset['train'].train_test_split(
        test_size=0.0005,
        seed=2357,
        shuffle=True,
    )
    split_dataset['val'] = split_dataset.pop('test')
    print(split_dataset)

    def process(example):
        # example 是 OpenWebText 里的一篇网页文本，字段 example['text'] 是原始字符串。
        # encode_ordinary 会把字符串切成 GPT-2 BPE token id。
        # 对 OpenWebText 这种多文档数据，官方会在每篇文档末尾加 eot_token，告诉模型“这篇结束”。
        # 否则两篇毫不相干的网页会在 train.bin 里硬连起来，模型会误以为它们是连续上下文。
        ids = enc.encode_ordinary(example['text'])
        ids.append(enc.eot_token)
        return {'ids': ids, 'len': len(ids)}

    print('开始 GPT-2 BPE 分词。')
    tokenized = split_dataset.map(
        process,
        remove_columns=['text'],
        desc='tokenizing the splits',
        num_proc=num_proc,
    )

    bin_paths = {}
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = data_dir / f'{split}.bin'
        arr = np.memmap(filename, dtype=np.uint16, mode='w+', shape=(arr_len,))
        total_batches = 1024
        idx = 0

        print()
        print(f'开始写入 {filename}')
        print(f'{split} token 总数:', f'{int(arr_len):,}')

        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename.name}'):
            batch = dset.shard(
                num_shards=total_batches,
                index=batch_idx,
                contiguous=True,
            ).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        arr.flush()
        bin_paths[split] = filename

    return bin_paths['train'], bin_paths['val']


prepared_bins = find_prepared_openwebtext_bins()

if prepared_bins is not None:
    train_bin_path, val_bin_path = prepared_bins
    print()
    print('发现 Kaggle Input 中已有 OpenWebText bin 文件，直接读取：')
    print('train.bin:', train_bin_path)
    print('val.bin:', val_bin_path)
elif PREPARE_OPENWEBTEXT_FROM_HF:
    train_bin_path, val_bin_path = prepare_openwebtext_from_hf()
else:
    raise FileNotFoundError(
        '没有在 /kaggle/input 中找到 train.bin / val.bin。\n'
        '推荐先在 Kaggle Add Input 里添加已经准备好的 OpenWebText 数据集。\n'
        '如果你确认空间和时间足够，也可以把 PREPARE_OPENWEBTEXT_FROM_HF 改成 True，按官方脚本现场下载预处理。'
    )


# 后面训练时不需要把完整 train.bin 读进内存。
# np.memmap 会建立“文件视图”，用到哪一段再读哪一段。
train_data = np.memmap(train_bin_path, dtype=np.uint16, mode='r')
val_data = np.memmap(val_bin_path, dtype=np.uint16, mode='r')

print()
print('OpenWebText 数据准备完成。')
print('train.bin:', train_bin_path, 'tokens:', f'{len(train_data):,}')
print('val.bin:', val_bin_path, 'tokens:', f'{len(val_data):,}')
print('训练集前 40 个 token id:')
print(train_data[:40].tolist())
print('解码前 40 个 token:')
print(enc.decode(train_data[:40].tolist()))


def encode(s):
    # 把字符串变成 GPT-2 BPE token id 列表。
    # 例如 'hello world' 可能不是 11 个字符 token，而是若干个常见子词片段 token。
    return enc.encode_ordinary(s)


def decode(ids):
    # 把 GPT-2 BPE token id 列表还原成字符串。
    # 注意 decode 是 tokenizer 层面的还原，不代表模型预测一定正确；它只负责把编号翻译回文本片段。
    return enc.decode(ids)


# 这一格产生的关键变量：
#   enc / encode / decode       GPT-2 BPE 分词器和辅助函数
#   vocab_size                  nanoGPT 从零训练 GPT-2 路线默认词表大小，50304
#   train_bin_path / val_bin_path
#   train_data / val_data       np.memmap 形式的 OpenWebText token 序列
#
# 下一格 Cell 3 会讲：
#   如何从 train_data 中随机取一批连续 token，
#   构造 x 和 y，让模型学习“看到前文，预测下一个 token”。


In [ ]:
# 第 3 个 cell：实现 get_batch，构造语言模型的 x 和 y
#
# 这一格对应 nanoGPT 官方源码 train.py 里的 get_batch(split)。
#
# 语言模型训练的核心任务非常朴素：
#   给模型看一段前文 x，让它预测每个位置的下一个 token，也就是 y。
#
# 假设原始 token 序列是：
#   [10, 20, 30, 40, 50]
# 如果 block_size = 4，那么：
#   x = [10, 20, 30, 40]
#   y = [20, 30, 40, 50]
#
# 这不是随便错开一位，而是在告诉模型：
#   看到 10，应该预测 20
#   看到 10,20，应该预测 30
#   看到 10,20,30，应该预测 40
#   看到 10,20,30,40，应该预测 50
#
# 后面算 loss 的时候，就是把模型每个位置预测出来的概率分布，
# 和这里的 y 逐个对答案。


# GPT-2 的默认上下文长度是 1024。
# 也就是说，模型每次最多看前面 1024 个 token 来预测后面的 token。
# 这里 batch_size 先设小一点，方便在 Kaggle 上观察输出，后面训练时可以再调大。
block_size = 1024
batch_size = 4


def get_batch(split):
    # 这个函数就是 nanoGPT train.py 里 get_batch 的教学展开版。
    # 它不负责“理解文本”，只负责从一条很长的 token 河流里随机截取训练样本。
    # 随机截取的好处是：每次训练都看到不同位置的片段，不会只按固定顺序背前几段数据。
    # split 决定从训练集还是验证集取样。
    # train_data / val_data 是上一格得到的 np.memmap，里面是一长串 GPT-2 BPE token id。
    data = train_data if split == 'train' else val_data

    if len(data) <= block_size:
        raise ValueError(
            f'{split} 数据太短：只有 {len(data)} 个 token，'
            f'但 block_size={block_size}。请换更大的数据，或减小 block_size。'
        )

    # ix 是一批随机起点。
    # 每个起点 i 都会切出一段长度为 block_size 的 x，
    # 再切出一段向右平移一位的 y。
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x_np = np.stack([
        np.array(data[int(i) : int(i) + block_size], dtype=np.int64)
        for i in ix
    ])
    y_np = np.stack([
        np.array(data[int(i) + 1 : int(i) + 1 + block_size], dtype=np.int64)
        for i in ix
    ])

    x = torch.from_numpy(x_np)
    y = torch.from_numpy(y_np)

    # 如果真能用 CUDA，就把 batch 搬到 GPU。
    # pin_memory + non_blocking 是官方 train.py 里的加速写法：
    # 它让 CPU 到 GPU 的数据搬运更顺畅。
    if device == 'cuda':
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x = x.to(device)
        y = y.to(device)

    return x, y, ix


xb, yb, ix = get_batch('train')

print('随机起点 ix:', ix.tolist())
print('x shape:', tuple(xb.shape))
print('y shape:', tuple(yb.shape))
print('x dtype/device:', xb.dtype, xb.device)
print('y dtype/device:', yb.dtype, yb.device)


# 看第一条样本的前 40 个 token。
# x 和 y 应该刚好错开一位。
sample_row = 0
show_tokens = 40

x_head = xb[sample_row, :show_tokens].detach().cpu().tolist()
y_head = yb[sample_row, :show_tokens].detach().cpu().tolist()

print()
print('第一条样本 x 的前 40 个 token id:')
print(x_head)
print('第一条样本 y 的前 40 个 token id:')
print(y_head)

print()
print('把 x 前 40 个 token 解码成文字：')
print(decode(x_head))
print('把 y 前 40 个 token 解码成文字：')
print(decode(y_head))


# 更细一点看：每个位置到底在学什么？
# 下面打印前 8 个位置。
# 左边是模型能看到的上下文，右边是它该预测的下一个 token。
print()
print('前 8 个训练小任务：')
for t in range(8):
    context_ids = xb[sample_row, : t + 1].detach().cpu().tolist()
    target_id = int(yb[sample_row, t].detach().cpu().item())
    context_text = decode(context_ids)
    target_text = decode([target_id])
    print(f'看到 {context_text!r} -> 预测下一个 token {target_text!r}  token_id={target_id}')


# 这一格产生的关键变量：
#   block_size      每条样本的上下文长度，GPT-2 默认是 1024
#   batch_size      每次并行取多少条样本
#   get_batch       从 train_data / val_data 随机取 x/y 的函数
#   xb / yb         一个实际训练 batch，可以直接喂给后面的模型
#


In [ ]:
# 第 4 个 cell：实现单头 causal self-attention
#
# 现在开始进入 Transformer 的核心：self-attention。
# 这一格只写“单头”，先把一个注意力头到底怎么算讲清楚。
#
# causal 的意思是“有因果顺序”：
#   第 t 个位置只能看 0..t，不能看 t+1..T-1。
# 这就是 GPT 能训练 next-token prediction 的关键。


class SingleHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout=0.0, bias=False):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=bias)
        self.query = nn.Linear(n_embd, head_size, bias=bias)
        self.value = nn.Linear(n_embd, head_size, bias=bias)
        self.dropout = nn.Dropout(dropout)

        # tril 是下三角矩阵。1 表示允许看，0 表示不允许看。
        # 例如 T=4 时：
        #   [[1,0,0,0],
        #    [1,1,0,0],
        #    [1,1,1,0],
        #    [1,1,1,1]]
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        # x: B x T x C
        B, T, C = x.shape

        # Q/K/V 都来自同一个 x，所以叫 self-attention。
        # q 表示“我当前位置想找什么信息”。
        # k 表示“我这个位置能被别人用什么特征找到”。
        # v 表示“如果别人关注我，我真正交出去的信息”。
        q = self.query(x)  # B x T x head_size
        k = self.key(x)    # B x T x head_size
        v = self.value(x)  # B x T x head_size

        # q @ k^T: B x T x hs @ B x hs x T -> B x T x T
        # att[b, i, j] 表示第 b 条样本里，位置 i 对位置 j 的关注分数。
        att = q @ k.transpose(-2, -1)
        att = att * (1.0 / math.sqrt(k.size(-1)))

        # 把未来位置的分数设成 -inf，softmax 后概率就是 0。
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        # 用注意力概率对 v 做加权求和。
        # att: B x T x T，v: B x T x hs，out: B x T x hs。
        out = att @ v
        return out


attention_demo_x = torch.randn(2, 8, 64, device=device)
single_head = SingleHeadCausalSelfAttention(n_embd=64, head_size=16, block_size=8).to(device)
attention_demo_y = single_head(attention_demo_x)
print('single-head input shape:', tuple(attention_demo_x.shape))
print('single-head output shape:', tuple(attention_demo_y.shape))


In [ ]:
# 第 5 个 cell：实现多头注意力
#
# 单头注意力只有一个“观察角度”。
# 多头注意力就是让多个头并行工作：
#   有的头可能学指代关系，有的头可能学短语搭配，有的头可能学标点/换行节奏。
#
# 这一格写成接近 nanoGPT model.py 的版本：
#   c_attn 一次性算出所有 head 的 Q/K/V
#   reshape 成 B x n_head x T x head_size
#   做 causal attention
#   再拼回 B x T x C，并经过 c_proj 输出投影


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.head_size = config.n_embd // config.n_head

        # c_attn 一次性生成 Q/K/V，所以输出维度是 3 * n_embd。
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)

        # c_proj 是多头注意力的输出投影。
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size))
                                      .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        # self.c_attn(x) 一次性算出 Q、K、V 三份向量。
        # 输入 x:        B x T x C
        # c_attn(x):     B x T x 3C
        # split 后 q/k/v: B x T x C
        # 这样写比三个 Linear 更紧凑，也更接近 GPT-2 官方实现。
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # 原来 q/k/v 都是：B x T x C，比如 B x T x 768。
        # view 后变成：B x T x n_head x head_size，比如 B x T x 12 x 64。
        # transpose(1, 2) 后变成：B x n_head x T x head_size，比如 B x 12 x T x 64。
        # 这样每个 head 就能独立在自己的 64 维小空间里算注意力。
        # 注意：view 不会改变数据本身，只是换一种形状解释同一批数。
        # transpose(1, 2) 是交换第 1 维和第 2 维，把 head 维提前，方便每个 head 并行算注意力。
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            # is_causal=True 会自动应用“不能看未来”的遮罩。
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v,
                attn_mask=None,
                dropout_p=self.dropout if self.training else 0,
                is_causal=True,
            )
        else:
            # q @ k.transpose(-2, -1):
            #   B x n_head x T x hs @ B x n_head x hs x T -> B x n_head x T x T
            # 这里得到的 att 不是“行列式”，而是矩阵乘法结果。
            # att[b,h,i,j] 表示第 b 条样本、第 h 个头中，位置 i 对位置 j 的匹配分数。
            # 除以 sqrt(head_size) 是缩放：避免 head_size 较大时点积分数过大，softmax 变得过尖。
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            # 下三角 mask 会把未来位置置为 -inf。
            # softmax 后这些位置概率变成 0，于是第 i 个 token 不能偷看 i 后面的 token。
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            # softmax 让每一行变成概率分布：当前位置应该从过去每个位置拿多少信息。
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            # att @ v 是按注意力概率对 value 做加权求和。
            # 如果某个位置 j 的注意力概率大，它的 v[j] 就会更多流入当前位置 i。
            y = att @ v

        # y: B x n_head x T x head_size -> B x T x n_head x head_size -> B x T x C。
        # contiguous 会在需要时重新整理内存，让后面的 view 能按连续内存安全读取。
        # transpose 常常只改变 stride，不一定真正搬数据；contiguous 会在需要时新分配连续内存并拷贝。
        # 所以后面的 view(B,T,C) 才能安全地把多个 head 拼回一个 C 维向量。
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


@dataclass
class AttentionDemoConfig:
    block_size: int = 8
    vocab_size: int = 100
    n_layer: int = 1
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.0
    bias: bool = False


multi_head = CausalSelfAttention(AttentionDemoConfig()).to(device)
multi_y = multi_head(attention_demo_x)
print('multi-head input shape:', tuple(attention_demo_x.shape))
print('multi-head output shape:', tuple(multi_y.shape))


In [ ]:
# 第 6 个 cell：实现 MLP 和 Block
#
# 一个 GPT Block = 预归一化 + 多头因果自注意力 + 残差连接 + 预归一化 + MLP + 残差连接。
#
# 对应 nanoGPT model.py：
#   LayerNorm
#   MLP
#   Block


class LayerNorm(nn.Module):
    """LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False."""

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        # input 常见形状：B x T x C。
        # LayerNorm 只在最后一维 C 上归一化，不混合 batch，也不混合不同 token 位置。
        # 对每个 b,t：
        #   mean[b,t] = input[b,t,:].mean()
        #   var[b,t]  = input[b,t,:].var(unbiased=False)
        #   y[b,t,c]  = (input[b,t,c] - mean[b,t]) / sqrt(var[b,t] + 1e-5) * weight[c] + bias[c]
        # 直觉类比：每个 token 的 C 个特征像一组分数，LayerNorm 先把这一组分数拉到稳定尺度，
        # 再交给 attention/MLP 处理。这样深层网络不容易因为数值忽大忽小而训练不稳。
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        # GPT-2 的前馈网络通常先把维度放大 4 倍，再压回 n_embd。
        # attention 负责“从上下文拿信息”，MLP 负责“对每个位置的信息做非线性加工”。
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        # GELU 是激活函数。精确定义是 GELU(x)=x*Phi(x)，Phi 是标准正态分布的累积分布函数。
        # 直觉：它不像 ReLU 那样简单地小于 0 全砍掉，而是更平滑地决定哪些信号通过多少。
        # 没有 GELU 这类非线性，多层 Linear 叠在一起本质仍是一个 Linear，表达能力会弱很多。
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        # 第一次残差：attention 在原信息基础上补充“上下文交流后的信息”。
        # self.ln_1(x) 先归一化，这是 pre-norm 写法；深层 GPT 更稳定。
        # x + attention(...) 表示不要让 attention 完全重写 x，而是在原稿上追加上下文修改意见。
        x = x + self.attn(self.ln_1(x))

        # 第二次残差：MLP 在 attention 处理后的信息基础上继续加工。
        # 所以一个 Block 里有两条残差路：attention 一条，MLP 一条。
        # 这和 illustrated-gpt2 里“每个 Transformer block 内部先注意力、再前馈网络”的图是对应的。
        x = x + self.mlp(self.ln_2(x))
        return x


block_demo = Block(AttentionDemoConfig()).to(device)
block_y = block_demo(attention_demo_x)
print('block input shape:', tuple(attention_demo_x.shape))
print('block output shape:', tuple(block_y.shape))


In [ ]:
# 第 7 个 cell：组装 GPTConfig 和 GPT
#
# 现在把前面的零件拼成完整 GPT：
#   token embedding + position embedding + 多层 Block + 最终 LayerNorm + lm_head
#
# 这一格是后面训练和生成共同使用的核心模型。
# 结构对照 nanoGPT model.py，但注释按从零理解来写。

import inspect


# GPTConfig 可以理解成“这台 GPT 机器的规格说明书”。
# 后面的 GPT 类不会把层数、头数、向量维度写死，而是从 config 里读取。
# 这样同一份 GPT 代码可以组装 GPT-2 small、medium、large，或者你自己的小模型。
@dataclass
class GPTConfig:
    # block_size：上下文窗口长度，也就是模型一次最多能看多少个 token。
    # GPT-2 默认是 1024。超过这个长度时，generate 会只保留最后 1024 个 token。
    # 它会决定位置嵌入表 wpe 的行数，也会影响注意力矩阵 T x T 的大小。
    block_size: int = 1024

    # vocab_size：词表大小，也就是模型一共认识多少种 token id。
    # GPT-2 BPE 原始词表是 50257；nanoGPT 从零训练时常用 50304，是为了把维度补齐到更适合 GPU 计算。
    # 它决定输入 embedding 表 wte 的行数，也决定 lm_head 最后要给多少个 token 打分。
    vocab_size: int = 50304

    # n_layer：Transformer Block 堆叠多少层。
    # 一层 Block 大致包含：LayerNorm -> causal self-attention -> 残差 -> LayerNorm -> MLP -> 残差。
    # GPT-2 small / 124M 使用 12 层；层数越多，表达能力越强，但训练更慢、显存更多。
    n_layer: int = 12

    # n_head：每一层注意力里有多少个 head。
    # 多头可以理解成多个观察角度并行看上下文，例如有的头看指代，有的头看局部搭配。
    # 必须满足 n_embd % n_head == 0，因为每个 head 分到 head_size = n_embd // n_head 维。
    n_head: int = 12

    # n_embd：每个 token 的隐藏向量宽度，也叫 embedding dimension / hidden size。
    # GPT-2 small 是 768。比如 idx 形状是 B x T，进入 wte 后就变成 B x T x 768。
    # 这里 768 / 12 = 64，所以每个 attention head 处理 64 维。
    n_embd: int = 768

    # dropout：训练时随机丢掉一部分特征的概率，用来降低过拟合。
    # 大规模 OpenWebText 预训练默认 0.0；小数据微调时常设 0.1 左右。
    # 注意 model.eval() 推理时 Dropout 会自动关闭。
    dropout: float = 0.0

    # bias：Linear 和 LayerNorm 里是否使用 bias 参数。
    # bias=True 时线性层是 y = xW + b；bias=False 时是 y = xW。
    # nanoGPT 从零训练默认 False；如果加载 OpenAI GPT-2 原版权重，通常需要 True 才能和权重结构对上。
    bias: bool = False


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # 权重共享：输入 token embedding 表和输出词表分类器共享同一套参数。
        # 这不是复制，而是让两边指向同一个 Parameter。
        # 输入时：wte.weight[token_id] 把 token 编号查成向量。
        # 输出时：lm_head 用同一张表和隐藏向量做匹配，给每个 token 打分。
        # 好处是参数更少，也让“理解 token 的空间”和“输出 token 的空间”天然对齐。
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)

        # c_proj 出现在 attention 的 self.attn.c_proj，以及 MLP 的 self.mlp.c_proj。
        # 对 GPT-2 small 的 12 层来说，会有 12 个 attention c_proj 和 12 个 MLP c_proj。
        # 这些投影都在残差分支上，很多层一路相加；初始化稍小可以让训练初期更稳。
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        print('number of parameters: %.2fM' % (self.get_num_params() / 1e6,))

    def get_num_params(self, non_embedding=True):
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx 是模型真正读进去的 token id，形状 B x T。
        # targets 是训练答案，形状 B x T；它只在最后算 loss，不进入 attention。
        #
        # 训练时调用 model(idx, targets)：
        #   idx     = 当前片段，例如 [10, 20, 30, 40]
        #   targets = 右移一位的答案，例如 [20, 30, 40, 50]
        # 推理时调用 model(idx)：
        #   没有 targets，因为真实下一个 token 还不知道，模型正在猜它。
        #
        # 非常关键：targets 不会进入 wte、wpe、attention、MLP。
        # 模型只读 idx；targets 只在最后 cross_entropy 里当标准答案。
        # 因此训练时模型没有偷看答案，防偷看靠 attention 的 causal mask 完成。
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f'Cannot forward sequence of length {t}, block size is only {self.config.block_size}'

        # pos 是位置编号 [0, 1, 2, ..., t-1]。
        # token id 只说明“词是谁”，pos 说明“词站在哪里”。
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        # tok_emb: B x T x C。每个 token id 查到一条 C 维语义向量。
        # pos_emb: T x C。每个位置查到一条 C 维位置向量，随后广播到 B x T x C。
        tok_emb = self.transformer.wte(idx)  # B x T x C
        pos_emb = self.transformer.wpe(pos)  # T x C
        # token 信息 + 位置信息，形状仍是 B x T x C。
        x = self.transformer.drop(tok_emb + pos_emb)

        for block in self.transformer.h:
            # 每个 block 先让 token 通过 masked self-attention 交换上下文信息，
            # 再通过 MLP 做逐位置的非线性加工。
            # 虽然 T 个位置是并行算的，但 causal mask 保证第 t 个位置只能看 0..t。
            x = block(x)
        # 最后一层 LayerNorm 把每个位置的隐藏向量稳定到适合输出分类的尺度。
        x = self.transformer.ln_f(x)

        if targets is not None:
            # 训练/验证：每个位置都有 next-token 答案，所以对所有 T 个位置算 logits。
            # 输入 x:      B x T x C
            # 输出 logits: B x T x V
            # logits[b,t,v] 表示第 b 条样本第 t 个位置，模型认为下一个 token 是 v 的原始分数。
            logits = self.lm_head(x)  # B x T x V
            # cross_entropy 要求 N x V 的分数和 N 个标签，所以把 B*T 个位置摊平：
            #   logits.view(-1, V): B*T x V
            #   targets.view(-1):   B*T
            # 这等价于一次性做 B*T 道“预测下一个 token”的分类题。
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # 推理/生成：只需要最后一个位置预测下一个 token。
            # 例如上下文是 [10,20,30,40]，现在只需要位置 3 的输出去预测第 5 个 token。
            # 位置 0/1/2 的输出分别是在预测 20/30/40，这些历史答案已经在上下文里了。
            # x[:, [-1], :] 使用 [-1] 而不是 -1，是为了保留 time 维度，输出仍是 B x 1 x V。
            # nanoGPT 这里没有 KV cache，所以 Transformer block 仍会重新算整段 idx；
            # 这个小优化只省掉前面 T-1 个位置的 lm_head 词表投影。
            logits = self.lm_head(x[:, [-1], :])  # B x 1 x V
            loss = None

        return logits, loss

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # optimizer 需要知道哪些参数应该做 weight decay，哪些不应该。
        # weight decay 可以理解为“别让权重变得过大”的约束，常用于防止过拟合。
        # 但 bias、LayerNorm 的 weight/bias 这类一维参数通常不做 decay，
        # 因为它们主要负责平移/缩放数值尺度，强行衰减反而可能影响稳定性。
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        # dim >= 2 的通常是矩阵：Linear weight、Embedding weight。
        # dim < 2 的通常是一维参数：bias、LayerNorm weight。
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        print(f'num decayed parameter tensors: {len(decay_params)}, with {sum(p.numel() for p in decay_params):,} parameters')
        print(f'num non-decayed parameter tensors: {len(nodecay_params)}, with {sum(p.numel() for p in nodecay_params):,} parameters')

        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        # fused AdamW 是 PyTorch/CUDA 的融合实现，数学上还是 AdamW，但多个小操作融合成更高效的 GPU kernel。
        print(f'using fused AdamW: {use_fused}')
        return torch.optim.AdamW(
            optim_groups,
            lr=learning_rate,
            betas=betas,
            **(dict(fused=True) if use_fused else dict()),
        )

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd // cfg.n_head, cfg.block_size
        flops_per_token = 6 * N + 12 * L * H * Q * T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        flops_achieved = flops_per_iter * (1.0 / dt)
        flops_promised = 312e12
        return flops_achieved / flops_promised

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            # idx 是当前已经有的上下文 token，形状 B x 当前长度。
            # 如果当前长度超过 block_size，只保留最后 block_size 个 token。
            # 因为模型训练时最多只学会看 block_size 长度的上下文。
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # 推理时不传 targets，所以会走 forward 里的 targets=None 分支。
            # 返回的 logits 只包含最后一个位置的词表分数：B x 1 x V。
            logits, _ = self(idx_cond)
            # 取最后一个位置，并除以 temperature。
            # temperature < 1 会让高分 token 更突出，输出更保守；temperature > 1 会更随机。
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                # top_k 只保留分数最高的 k 个 token，其余设为 -inf。
                # softmax 后，被设为 -inf 的 token 概率为 0，可以减少特别离谱的采样。
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            # softmax 把 logits 变成概率分布，然后 multinomial 按概率抽样一个 token。
            # 不是每次都取最大值，所以同一个 prompt 可以生成不同文本。
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            # 把新 token 拼回上下文，下一轮再用更长的 idx 继续预测。
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


gpt_config = GPTConfig(
    block_size=block_size,
    vocab_size=vocab_size,
    n_layer=12,
    n_head=12,
    n_embd=768,
    dropout=0.0,
    bias=False,
)

model = GPT(gpt_config).to(device)
model.eval()

smoke_x = xb[:1, :8].to(device)
smoke_y = yb[:1, :8].to(device)
with torch.no_grad():
    smoke_logits, smoke_loss = model(smoke_x, smoke_y)

print('smoke_x shape:', tuple(smoke_x.shape))
print('smoke_logits shape:', tuple(smoke_logits.shape))
print('smoke_loss:', float(smoke_loss.item()))


In [ ]:
# 第 8 个 cell：写训练循环，跑多轮训练，并启用 notebook 可用的多卡优化
#
# 这一格对照 nanoGPT train.py 的训练主循环，但先不讲 checkpoint。
# 现在只关注训练本身：
#   学习率调度 -> forward -> loss -> backward -> 梯度裁剪 -> optimizer.step -> 定期评估
#
# 多卡说明：
#   nanoGPT 官方正式多卡使用 torchrun + DistributedDataParallel(DDP)。
#   普通 Kaggle notebook cell 不是 torchrun 启动的，所以不能自然进入官方 DDP 分支。
#   这里用 nn.DataParallel 在 notebook 里利用多张 GPU 做演示加速；模型结构和训练目标不变。

import gc
import sys
import time
from contextlib import nullcontext


def cleanup_training_memory():
    # Notebook 里如果中断了一次训练，Python 进程不会重启。
    # 这意味着上一次 Cell 8 创建的 raw_model / train_model / optimizer / loss / X / Y 等变量，
    # 以及 IPython 的最后输出 _ / Out、异常 traceback，都可能还在引用旧 CUDA 张量。
    # 只调用 torch.cuda.empty_cache() 不会释放这些仍被引用的显存。
    # 所以重新运行训练 cell 前，先尽量显式删除这些旧对象。
    names_to_delete = [
        'model', 'raw_model', 'train_model', 'optimizer', 'scaler',
        'logits', 'loss', 'losses', 'X', 'Y', 'mfu', 'lr', 'dt',
        'gpt_config', 'compiled_model', 'unoptimized_model',
        # 下面这些是前面 cell 或旧版本 notebook 里可能残留的 CUDA 对象。
        'tiny_model', 'tiny_optimizer', 'tiny_x', 'tiny_y', 'loss_after',
        'single_head', 'attention_demo_x', 'attention_demo_y',
        'multi_head', 'multi_y', 'block_demo', 'block_y',
        'smoke_x', 'smoke_y', 'smoke_logits', 'smoke_loss',
        'sample_model', 'sample_checkpoint', 'y', 'probs', 'idx_next',
    ]
    deleted = []
    for name in names_to_delete:
        if name in globals():
            del globals()[name]
            deleted.append(name)

    # IPython/Jupyter 会把最后一个表达式结果放到 _、__、___，并把历史输出放到 Out/_oh。
    # 如果你曾经让某个 CUDA Tensor 或模型作为 cell 最后一行输出，它可能被这些缓存引用。
    try:
        ip = get_ipython()
        for name in ['_', '__', '___']:
            if name in ip.user_ns:
                del ip.user_ns[name]
                deleted.append(name)
        for name in ['Out', '_oh']:
            obj = ip.user_ns.get(name, None)
            if hasattr(obj, 'clear'):
                obj.clear()
                deleted.append(name + '.clear()')
    except Exception:
        pass

    # OOM 异常的 traceback 也可能暂时引用局部变量，例如 loss、logits、模型等。
    # 清掉 sys.last_* 后再 gc，可以减少“明明 del 了但还被异常栈引用”的情况。
    for attr in ['last_type', 'last_value', 'last_traceback']:
        if hasattr(sys, attr):
            try:
                setattr(sys, attr, None)
            except Exception:
                pass

    # 如果上一次启用了 torch.compile，Dynamo/Inductor 也可能持有编译图相关缓存。
    try:
        torch._dynamo.reset()
        deleted.append('torch._dynamo.reset()')
    except Exception:
        pass

    # gc.collect() 触发 Python 垃圾回收，清掉已经没有引用的对象。
    # empty_cache() 释放 PyTorch 缓存池里暂时不用的显存。
    # ipc_collect() 清理 CUDA IPC 相关的缓存，notebook 多次中断/重跑时也有帮助。
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        gc.collect()
        torch.cuda.empty_cache()

    print('cleanup deleted:', deleted if deleted else 'nothing')
    if torch.cuda.is_available():
        for gpu_id in range(torch.cuda.device_count()):
            allocated = torch.cuda.memory_allocated(gpu_id) / 1024**3
            reserved = torch.cuda.memory_reserved(gpu_id) / 1024**3
            print(f'GPU {gpu_id} after cleanup: allocated={allocated:.2f} GiB, reserved={reserved:.2f} GiB')
        print('说明：allocated 是仍被活跃张量占用的显存；reserved 是 PyTorch 缓存池保留的显存。')
        print('如果 allocated 很低但 reserved 仍高，通常是缓存/碎片；若仍 OOM，最干净的方法是 Restart Kernel。')


# 每次运行 Cell 8 时，先清理上一次训练或中断后残留的显存引用。
cleanup_training_memory()


gradient_accumulation_steps = 5 * 8
batch_size = 12
block_size = 1024

learning_rate = 6e-4
weight_decay = 1e-1
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0

warmup_iters = 2000
lr_decay_iters = 600000
min_lr = 6e-5
decay_lr = True

# 为了让第 9 格 generate 的输出更像正常文字，这里默认多跑一些 iteration。
# 注意：这是从零训练 GPT-2 结构，不是微调现成 GPT-2；50 步只能让 loss 有一点下降，
# 真正流畅的语言能力仍然需要大量 token 和长时间训练。官方 GPT-2 124M 复现是 600000 步量级。
# 如果你只是想快速检查代码，把 train_iters 改回 3；如果你想多训一点，可以改成 100、500、1000。
train_iters = 500

# eval 会额外跑 train/val batch，只用于观察 loss，不更新参数。
# 训练步数变多后，不要每一步都 eval，否则会浪费很多时间。
eval_interval = 10
eval_iters = 5
log_interval = 1

# torch.compile 可以把 PyTorch 模型编译成更高效的执行图，nanoGPT 官方 train.py 默认 compile=True。
# 但它第一次运行会有明显编译开销；如果只跑几十步，可能“编译花的时间比训练省下的时间还多”。
# 建议：
#   1. 先保持 False，确认数据、模型、loss 都正常。
#   2. 当 train_iters 提到 500、1000 或更长训练时，再改成 True。
#   3. 普通 notebook 多卡这里用的是 nn.DataParallel，和 torch.compile 的收益/兼容性不如官方 DDP 稳定。
#      所以下面代码只在单卡或未启用多卡时编译 train_model。
compile_model = False

# 多卡开关。
# True：如果当前环境有多张 CUDA GPU，就用 nn.DataParallel 做 notebook 版多卡训练。
# False：即使机器有多张 GPU，也只用单卡训练，方便调试、对比速度、或者配合 torch.compile。
# 注意：这里的多卡是 notebook 里容易运行的 DataParallel，不是 nanoGPT 官方 torchrun + DDP。
use_multi_gpu = True


device_type = 'cuda' if 'cuda' in device else 'cpu'
if device_type == 'cuda':
    dtype = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
else:
    dtype = 'float32'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

scaler_enabled = (dtype == 'float16' and device_type == 'cuda')
try:
    scaler = torch.amp.GradScaler(device_type, enabled=scaler_enabled)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)

print('device:', device)
print('device_type:', device_type)
print('dtype:', dtype)
print('cuda device count:', torch.cuda.device_count() if torch.cuda.is_available() else 0)
print('GradScaler enabled:', scaler_enabled)


# Cell 7 里已经做过 smoke test。这里会创建一个干净模型用于训练。
# 旧训练对象已经在本格开头的 cleanup_training_memory() 里释放过。

gpt_config = GPTConfig(
    block_size=block_size,
    vocab_size=vocab_size,
    n_layer=12,
    n_head=12,
    n_embd=768,
    dropout=0.0,
    bias=False,
)
raw_model = GPT(gpt_config).to(device)
train_model = raw_model

use_data_parallel = device_type == 'cuda' and use_multi_gpu and torch.cuda.device_count() > 1
print('use_multi_gpu requested:', use_multi_gpu)
print('effective DataParallel enabled:', use_data_parallel)

if compile_model and not use_data_parallel:
    # 长训练时可以打开这个分支。
    # 编译不改变模型数学结构，只是让 PyTorch 尽量把计算图优化得更快。
    print('compiling the model with torch.compile...')
    # 注意：这里不覆盖 raw_model。
    # raw_model 保留原始 GPT 对象，后面 configure_optimizers、clip_grad、generate 都继续用它。
    # train_model 是用于 forward/backward 的编译版本，参数仍然来自 raw_model。
    train_model = torch.compile(raw_model)
elif compile_model and use_data_parallel:
    # 为了 notebook 稳定性，这里不把 torch.compile 和 nn.DataParallel 叠在一起。
    # 真正追求官方多卡速度时，推荐用 torchrun + DistributedDataParallel + torch.compile。
    print('compile_model=True, but skipped torch.compile because notebook DataParallel is enabled.')

if use_data_parallel:
    # DataParallel 会把一个 batch 按 batch 维切开，分到多张 GPU 上计算，再汇总到主卡。
    # 这比官方 DDP 简单，适合 notebook 演示；真正大规模训练仍建议 DDP。
    train_model = nn.DataParallel(raw_model)
    print('using nn.DataParallel on', torch.cuda.device_count(), 'GPUs')
else:
    print('using single-device training')

optimizer = raw_model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)


def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)


@torch.no_grad()
def estimate_loss():
    out = {}
    train_model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y, _ = get_batch(split)
            with ctx:
                logits, loss = train_model(X, Y)
                if loss.dim() > 0:
                    # DataParallel 下每张 GPU 可能返回一个 loss，取平均作为总 loss。
                    loss = loss.mean()
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    train_model.train()
    return out


tokens_per_iter = gradient_accumulation_steps * batch_size * block_size
total_demo_tokens = tokens_per_iter * train_iters
print(f'tokens per iteration: {tokens_per_iter:,}')
print(f'total demo training tokens: {total_demo_tokens:,}')

running_mfu = -1.0
train_model.train()

for iter_num in range(train_iters):
    t0 = time.time()

    # 每个 iteration 先根据当前步数设置学习率。
    # 前期 warmup 慢慢升高，避免刚开始随机权重时步子太大；后期 cosine decay 慢慢降低，方便收敛。
    lr = get_lr(iter_num) if decay_lr else learning_rate
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if iter_num % eval_interval == 0:
        # eval 不更新参数，只估计 train/val loss。
        # train loss 看模型是否能拟合训练数据；val loss 看它对没见过的数据是否也更会预测。
        losses = estimate_loss()
        print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # 清空上一轮梯度。
    # set_to_none=True 不是把旧梯度填 0，而是直接设为 None，通常更省显存、更快。
    optimizer.zero_grad(set_to_none=True)

    for micro_step in range(gradient_accumulation_steps):
        # 梯度累积：连续跑多个 micro-batch，只 backward 累积梯度，最后统一 optimizer.step。
        # 这样可以模拟更大的 batch，而不要求显存一次装下全部样本。
        X, Y, _ = get_batch('train')
        with ctx:
            logits, loss = train_model(X, Y)
            if loss.dim() > 0:
                loss = loss.mean()
            # 必须除以 gradient_accumulation_steps。
            # 否则 40 个 micro-batch 的梯度直接相加，等价于把学习率偷偷放大 40 倍。
            loss = loss / gradient_accumulation_steps

        # backward 会沿计算图从 loss 往回推，给每个可学习参数计算 .grad。
        # 如果使用 float16，scaler 会先放大 loss，减少小梯度下溢成 0 的风险。
        scaler.scale(loss).backward()

    if grad_clip != 0.0:
        # 梯度裁剪：如果梯度总范数太大，就按比例缩小。
        # 直觉是限制每次更新的最大步幅，防止某个 batch 把参数一步推飞。
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(raw_model.parameters(), grad_clip)

    # optimizer.step() 才是真正改参数。
    # scaler.update() 会根据这一步是否出现 inf/nan 调整下一次的缩放倍数。
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    dt = time.time() - t0
    if iter_num % log_interval == 0:
        lossf = loss.item() * gradient_accumulation_steps
        if iter_num >= 1 and device_type == 'cuda':
            mfu = raw_model.estimate_mfu(batch_size * gradient_accumulation_steps, dt)
            running_mfu = mfu if running_mfu < 0 else 0.9 * running_mfu + 0.1 * mfu
        print(f'iter {iter_num}: loss {lossf:.4f}, lr {lr:.8f}, time {dt*1000:.2f}ms, mfu {running_mfu*100:.2f}%')

print('training done')
model = raw_model


In [ ]:
# 第 9 个 cell：写 generate，输出验证
#
# 训练后做一次输出验证：
#   prompt -> GPT-2 BPE token id -> model.generate -> token id -> 文本
#
# 如果只训练了第 9 格默认的几轮，输出很可能仍然混乱。
# 这说明模型还没充分训练，不说明 generate 流程错了。


prompt = "The meaning of life is"
max_new_tokens = 80
temperature = 0.8
top_k = 200
num_samples = 3

model.eval()
start_ids = encode(prompt)
x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

print('prompt:', repr(prompt))
print('start_ids:', start_ids)
print('x shape:', tuple(x.shape))

with torch.no_grad():
    with ctx:
        for i in range(num_samples):
            y = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
            text = decode(y[0].tolist())
            print('\n--------------- sample', i + 1, '---------------')
            print(text)

model.train()
